In [12]:
import pandas as pd

era5 = pd.read_csv(
    "/kaggle/input/datasets/arkalibaig/karakoram-era5-climate-dataset/gb_weather_corrected.csv"
)
era5.head(40)

,date,era5_period,station,district,lat,lon,elevation_m,temperature_2m_max,temperature_2m_min,temperature_2m_mean,...,et0_fao_evapotranspiration,sunshine_duration,daylight_duration,cloud_cover_mean,sunrise,sunset,weather_code,bias_corrected,month,year
0,1990-01-01,pre_2016,Chilas,Diamer,35.42,74.1,1250,8.66,5.70,6.90,...,1.14,24072.38,35307.29,85.14,1990-01-01 07:12:00,1990-01-01 17:01:00,71,0,1,1990
1,1990-01-02,pre_2016,Chilas,Diamer,35.42,74.1,1250,8.36,5.50,6.70,...,1.24,26205.91,35342.99,87.14,1990-01-02 07:13:00,1990-01-02 17:02:00,71,0,1,1990
2,1990-01-03,pre_2016,Chilas,Diamer,35.42,74.1,1250,8.36,6.40,7.50,...,0.96,1818.02,35381.64,92.14,1990-01-03 07:13:00,1990-01-03 17:02:00,73,0,1,1990
3,1990-01-04,pre_2016,Chilas,Diamer,35.42,74.1,1250,8.76,5.00,7.80,...,1.20,33518.04,35423.17,71.14,1990-01-04 07:13:00,1990-01-04 17:03:00,73,0,1,1990
4,1990-01-05,pre_2016,Chilas,Diamer,35.42,74.1,1250,4.66,0.30,2.70,...,1.25,34727.86,35467.52,52.14,1990-01-05 07:13:00,1990-01-05 17:04:00,71,0,1,1990
5,1990-01-06,pre_2016,Chilas,Diamer,35.42,74.1,1250,6.76,4.20,5.70,...,0.97,872.29,35514.62,89.14,1990-01-06 07:13:00,1990-01-06 17:05:00,73,0,1,1990
6,1990-01-07,pre_2016,Chilas,Diamer,35.42,74.1,1250,8.16,5.50,6.60,...,1.31,33882.08,35564.44,44.14,1990-01-07 07:13:00,1990-01-07 17:06:00,71,0,1,1990
7,1990-01-08,pre_2016,Chilas,Diamer,35.42,74.1,1250,7.76,4.80,6.10,...,1.36,34723.30,35616.88,25.14,1990-01-08 07:13:00,1990-01-08 17:07:00,1,0,1,1990
8,1990-01-09,pre_2016,Chilas,Diamer,35.42,74.1,1250,6.36,2.00,4.40,...,1.33,34726.09,35671.91,20.14,1990-01-09 07:13:00,1990-01-09 17:07:00,2,0,1,1990
9,1990-01-10,pre_2016,Chilas,Diamer,35.42,74.1,1250,6.56,3.20,5.20,...,1.32,31131.11,35729.45,64.14,1990-01-10 07:13:00,1990-01-10 17:08:00,3,0,1,1990


In [13]:
# mean and std per station calculation & merge back onto main dataframe
station_stats = era5.groupby('station')['precipitation_sum'].agg(['mean', 'std'])

era5 = era5.merge(station_stats, on='station', suffixes=('', '_station'))

#threshold calculation
era5['extreme_threshold'] = era5['mean'] + 2 * era5['std']

#is_extreme label created 
era5['is_extreme'] = era5['precipitation_sum'] > era5['extreme_threshold']

era5['is_extreme'] = era5['precipitation_sum'] > era5['extreme_threshold']
era5['is_extreme'] = era5['is_extreme'].astype(int)  # convert True/False to 1/0

print(era5['is_extreme'].value_counts())
print(era5['is_extreme'].value_counts(normalize=True))  # shows as percentages



is_extreme
0    61054
1     2866
Name: count, dtype: int64
is_extreme
0    0.955163
1    0.044837
Name: proportion, dtype: float64


In [14]:
era5['is_extreme'].value_counts()

is_extreme
0    61054
1     2866
Name: count, dtype: int64

In [15]:
era5[['station', 'date']].head(10)

,station,date
0,Chilas,1990-01-01
1,Chilas,1990-01-02
2,Chilas,1990-01-03
3,Chilas,1990-01-04
4,Chilas,1990-01-05
5,Chilas,1990-01-06
6,Chilas,1990-01-07
7,Chilas,1990-01-08
8,Chilas,1990-01-09
9,Chilas,1990-01-10


In [16]:
# Lag features (grouped by station, so no cross-contamination)
era5['precip_hours_lag1'] = era5.groupby('station')['precipitation_hours'].shift(1)
era5['cloud_cover_lag1'] = era5.groupby('station')['cloud_cover_mean'].shift(1)
era5['cloud_cover_roll3'] = era5.groupby('station')['cloud_cover_mean'].transform(lambda x: x.rolling(3).mean())
era5['wind_speed_roll3'] = era5.groupby('station')['wind_speed_10m_max'].transform(lambda x: x.rolling(3).mean())

# Check for NaNs created by shift/rolling
print(era5[['precip_hours_lag1', 'cloud_cover_lag1', 'cloud_cover_roll3', 'wind_speed_roll3']].isna().sum())

precip_hours_lag1     5
cloud_cover_lag1      5
cloud_cover_roll3    10
wind_speed_roll3     10
dtype: int64


In [17]:
era5 = era5.dropna(subset=['precip_hours_lag1', 'cloud_cover_lag1', 'cloud_cover_roll3', 'wind_speed_roll3'])
print(era5.shape)

(63910, 40)


In [18]:
columns_to_drop = [
    'date', 'era5_period', 'weather_code', 'sunrise', 'sunset',
    'bias_corrected', 'station', 'district',
    'precipitation_sum', 'rain_sum', 'snowfall_sum',
    'mean', 'std', 'extreme_threshold'
]

era5 = era5.drop(columns=columns_to_drop)
print(era5.columns.tolist())
print(era5.shape)

['lat', 'lon', 'elevation_m', 'temperature_2m_max', 'temperature_2m_min', 'temperature_2m_mean', 'apparent_temperature_max', 'apparent_temperature_min', 'apparent_temperature_mean', 'precipitation_hours', 'wind_speed_10m_max', 'wind_speed_10m_mean', 'wind_gusts_10m_max', 'wind_direction_10m_dominant', 'shortwave_radiation_sum', 'et0_fao_evapotranspiration', 'sunshine_duration', 'daylight_duration', 'cloud_cover_mean', 'month', 'year', 'is_extreme', 'precip_hours_lag1', 'cloud_cover_lag1', 'cloud_cover_roll3', 'wind_speed_roll3']
(63910, 26)


In [19]:
train = era5[era5['year'] <= 2018]
test = era5[era5['year'] >= 2019]

X_train = train.drop(columns=['is_extreme'])
y_train = train['is_extreme']
X_test = test.drop(columns=['is_extreme'])
y_test = test['is_extreme']

print(X_train.shape, X_test.shape)
print(train['is_extreme'].mean(), test['is_extreme'].mean())

(52950, 25) (10960, 25)
0.04304060434372049 0.053558394160583944


In [31]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)  # note: raw X_train, not X_train_scaled, trees don't need scaling


RandomForestClassifier(class_weight='balanced', n_estimators=200, n_jobs=-1,
                       random_state=42)

In [32]:
y_pred_rf = rf.predict(X_test)
y_pred_proba_rf = rf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_rf))

              precision    recall  f1-score   support

           0       0.95      1.00      0.97     10373
           1       0.73      0.13      0.22       587

    accuracy                           0.95     10960
   macro avg       0.84      0.56      0.60     10960
weighted avg       0.94      0.95      0.93     10960



In [33]:
for t in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
    y_pred_t = (y_pred_proba_rf > t).astype(int)
    p = precision_score(y_test, y_pred_t)
    r = recall_score(y_test, y_pred_t)
    f1 = f1_score(y_test, y_pred_t)
    print(f"Threshold {t}: precision={p:.3f}, recall={r:.3f}, f1={f1:.3f}")

Threshold 0.2: precision=0.593, recall=0.532, f1=0.561
Threshold 0.3: precision=0.674, recall=0.359, f1=0.469
Threshold 0.4: precision=0.677, recall=0.218, f1=0.330
Threshold 0.5: precision=0.726, recall=0.131, f1=0.222
Threshold 0.6: precision=0.745, recall=0.060, f1=0.110
Threshold 0.7: precision=0.826, recall=0.032, f1=0.062


In [34]:
import pandas as pd

importances = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print(importances)

precipitation_hours            0.341047
cloud_cover_mean               0.136525
sunshine_duration              0.092899
cloud_cover_roll3              0.056670
precip_hours_lag1              0.053697
cloud_cover_lag1               0.031594
shortwave_radiation_sum        0.029310
wind_gusts_10m_max             0.027018
apparent_temperature_min       0.023236
temperature_2m_min             0.021300
daylight_duration              0.020865
et0_fao_evapotranspiration     0.016736
apparent_temperature_mean      0.015415
wind_direction_10m_dominant    0.014811
wind_speed_10m_mean            0.014384
wind_speed_roll3               0.014006
temperature_2m_mean            0.013169
temperature_2m_max             0.013009
wind_speed_10m_max             0.012989
apparent_temperature_max       0.012422
month                          0.009514
lat                            0.009371
year                           0.008714
elevation_m                    0.005946
lon                            0.005353
